## 4. 演習1: ファイルアクセスと画像の操作

この章では、バラの JPEG 画像を使用し、以下を学習します。
1. IDL でのファイルの指定方法とデータの読み込み方法
2. 画像データの基本操作（表示・切り取り・拡大縮小・回転）
3. 画像処理演習として、フィルタリング処理を行い画像内の赤い部分を抜き出す

> ※ 使用データは IDL インストールディレクトリ配下の `examples\data` にあるサンプルデータを利用します。

### 4.1 ファイルを指定する

データの読込みを行うには、まずどのファイルからデータを読み込むか指定する必要があります。

**直接指定する**: データファイルのパスとファイル名をクォーテーションで囲んで指定する、最も単純な方法です。


In [ ]:
file = 'C:\Program Files\NV5\ENVIxx\IDLxx\examples\data\Day.jpg'
PRINT, file

**ファイルアクセスルーチンを使用する**

- `FILE_SEARCH` 関数: 引数で指定したディレクトリ内のファイルを検索し、見つかったすべてのファイルパスを文字列型で返す
- `FILE_WHICH` 関数: 第一引数のディレクトリ内から第二引数のファイルを検索し、最初に見つかったファイルパスを返す（ディレクトリ省略時はシステム変数 `!PATH` を検索）
- `DIALOG_PICKFILE` 関数: ファイル選択ダイアログを表示し、対話的にファイルを指定


In [ ]:
file = FILE_SEARCH('*.tif')
PRINT, file
; boulder.tif
; examples.tif
; image.tif

file = FILE_WHICH('people.dat')
PRINT, file
; C:\Program Files\NV5\ENVIxx\IDLxx\examples\data\people.dat

In [ ]:
file = DIALOG_PICKFILE()   ; ダイアログで Day.jpg を選択
PRINT, file

> **Note**: `!` から始まる変数は IDL のシステム変数です（すべてのプログラムで使用できる定義済み変数）。詳細は `IDL > IDL Concepts > IDL Programming > Variables > System Variables` を参照。
> `!PATH` のデフォルトは `<IDL_DEFAULT>` という特殊トークンで、`lib` や `examples` を含みます。ユーザがパスを追加することも可能です（`IDL > ... > Environment Variables`）。

### 4.2 JPEG ファイルを読み込む

`READ_JPEG` プロシージャを使用し JPEG ファイル（バラの画像: `rose.jpg`）を読み込みます。


In [ ]:
file = 'rose.jpg'
READ_JPEG, file, rose
HELP, rose
; ROSE            BYTE      = Array[3, 227, 149]

> **Note**: IDL は一般的なデータフォーマットの読み込み/書き込みルーチンを多数持っています。詳細は `IDL > Routines (by topic) > Input/Output` を参照。

### 4.3 データを表示する

読み込んだ画像データを `IMAGE` 関数で表示します。引数には2次元配列（白黒）または3次元配列（いずれかの次元が3であれば RGB）を指定できます。


In [ ]:
i = IMAGE(rose)

> **便利機能: オートコンプリート** — エディタやターミナルで入力中に候補が自動的にポップアップします。表示されない場合は `Ctrl+Space` で明示的に呼び出すことができます。入力中の文字列から始まる変数・ルーチンの一覧が表示されます。

### 4.4 データを切り取る

IDL は画像を配列型で持つため、配列の一部を抜き出すだけで画像の一部を切り取れます。ここでは横 40-190、縦 10-120 を切り取ります。


In [ ]:
clip = rose[*, 40:190, 10:120]
i = IMAGE(clip)

> `*` は「すべて」、`:` は「範囲指定」を表すワイルドカードです（詳細: `IDL > Routines (alphabetical) > Routines: F > FILE_SEARCH > Supported Wildcards and Expansions`）。
>
> **便利機能: ホバーヘルプ** — 不明な関数・プロシージャの上にマウスカーソルを乗せるだけで、ドキュメントを含むヘルプがその場にポップアップ表示されます。

### 4.5 データを拡大・縮小する

- `CONGRID` 関数: 任意のサイズに変更
- `REBIN` 関数: 元のサイズの整数倍に変更


In [ ]:
zoomin = CONGRID(rose, 3, 500, 300)   ; 500x300 に拡大
zi = IMAGE(zoomin)

zoomout = CONGRID(rose, 3, 113, 74)    ; 約半分に縮小
zo = IMAGE(zoomout)

### 4.6 データを回転する

`ROTATE` 関数は2次元配列のみ対応のため、R（赤）成分のみを抜き出して使用します。


In [ ]:
red = rose[0, *, *]
HELP, red
; RED             BYTE      = Array[1, 227, 149]

red = REFORM(red)     ; サイズ1の次元を削除して2次元配列に変換
HELP, red
; RED             BYTE      = Array[227, 149]

rt = ROTATE(red, 7)   ; 7 = 上下反転（設定値は 0〜7）
i = IMAGE(rt)

> `REFORM` は配列全体のサイズを変えずに次元を変更するための関数です。`ROTATE` はデータそのものを並び替えるルーチンで、第二引数（0〜7）で回転・反転方法を指定します。

### 4.7 マルチプロットする

`LAYOUT` キーワード（`[列数, 行数, インデックス]`）を使用すると、1つのウィンドウに複数のプロットを表示できます。ここでは RGB カラー画像と、R・G・B 各成分のグレースケール画像を表示します。


In [ ]:
i = IMAGE(rose, LAYOUT=[2, 2, 1], TITLE='RGB')
i = IMAGE(REFORM(rose[0, *, *]), LAYOUT=[2, 2, 2], TITLE='Red', /CURRENT)
i = IMAGE(REFORM(rose[1, *, *]), LAYOUT=[2, 2, 3], TITLE='Green', /CURRENT)
i = IMAGE(REFORM(rose[2, *, *]), LAYOUT=[2, 2, 4], TITLE='Blue', /CURRENT)

> **便利機能: コマンド履歴** — 統合ターミナルで上下キーを押すと、入力したコマンド履歴をさかのぼって呼び出せます。

### 4.8 演習: マスクをかけて表示する

画像処理演習として、マスクを作成し画像をフィルタリングして赤い部分を抜き出します。

まず画像上でマウスクリックし、ウィンドウ右下に表示される X/Y 座標や RGB データ値を確認して、赤い部分の特徴を調べます。


In [ ]:
i = IMAGE(rose)

R のデータ値と G のデータ値の差分に着目し、関係演算子 `GE`（Greater Equal）を使用してマスクを作成します。差分が 40 以上のピクセルを 1、その他を 0 とします。


In [ ]:
red = rose[0, *, *]
green = rose[1, *, *]
msk = (FIX(red) - FIX(green)) GE 40
m = IMAGE(BYTSCL(REFORM(msk)))

> **Note**: `red`・`green` は BYTE 型（範囲 0-255）で負の値を格納できないため、`red - green` の演算前に `FIX` 関数で整数型にキャストする必要があります。データ型の取り扱いには注意しましょう。
> `BYTSCL` はデータを 0-255 の範囲にストレッチするルーチンです。

最後に、RGB それぞれのデータにマスクを適用してフィルタリングします。


In [ ]:
img = BYTARR(3, 227, 149)
img[0, *, *] = rose[0, *, *] * msk
img[1, *, *] = rose[1, *, *] * msk
img[2, *, *] = rose[2, *, *] * msk
i = IMAGE(img)

---

前へ: [IDL の基本知識](03_basics.ipynb) | 次へ: [演習2: 標高データの3D表示と等高線表示](05_exercise2_3d_and_contour.ipynb) | [目次に戻る](00_overview.ipynb)